<a href="https://colab.research.google.com/github/aceville360/VU_sankara-derrick_programming_practice_farm_operations/blob/main/daily_farm_operations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sankara Derrick (VU-BAD-2603-0551-DAY) - Python for Data Science - Coursework 2 (September 2026)
**Project:** Predicting Daily Egg Yield on a Layer Poultry Farm


## 1. Data Ingestion & Exploration (Task 1 & Task 2)
We load our daily farm dataset collected over 210 continuous days (March 1 to September 27, 2026) from our layer poultry farm in Uganda.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# Load the dataset
df = pd.read_csv("farm_operations_daily.csv")
print("Initial Dataset Shape:", df.shape)
df.head()


In [ ]:
# Summary statistics and missing values audit
df.info()
print("\n--- Missing Value Count ---")
print(df.isna().sum())
print("\n--- Summary Statistics (Raw Data) ---")
display(df.describe())

# Parse dates and sort chronologically
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# Correct the May 3rd clerical entry typo (2,000 kg -> 200 kg)
# Normal layers eat ~100-120g/day. 2000kg for 1948 birds is an impossible entry error.
outlier = df["Feed_Consumed_kg"] > 500
if outlier.any():
    for idx in df[outlier].index:
        date_str = df.loc[idx, "Date"].strftime("%Y-%m-%d")
        old_val = df.loc[idx, "Feed_Consumed_kg"]
        df.loc[idx, "Feed_Consumed_kg"] = 200.0
        print(f"Corrected clerical typo on {date_str}: {old_val} kg -> 200.0 kg")


## 2. Feature Engineering (Task 2d)
We engineer practical biological and temporal features:
- `Days_in_Lay`: Days since lay cycle started (maturity).
- `Feed_per_Bird_g`: Daily feed in grams per bird.
- `Lay_Rate_Pct`: Daily egg production percentage (Eggs / Birds * 100).
- `Lag Features (1, 2, 3, 7)`: Previous days' values to capture biological delay.
- `7-Day Rolling Averages`: Weekly trend using shift(1) to avoid lookahead leakage.


In [ ]:
data = df.copy()
data["Days_in_Lay"] = np.arange(len(data))
data["Feed_per_Bird_g"] = (data["Feed_Consumed_kg"] * 1000.0) / data["Flock_Size"]
data["Lay_Rate_Pct"] = (data["Eggs_Collected"] / data["Flock_Size"]) * 100.0

for l in [1, 2, 3, 7]:
    data[f"LayRate_Lag{l}"] = data["Lay_Rate_Pct"].shift(l)
    data[f"Eggs_Lag{l}"] = data["Eggs_Collected"].shift(l)
    data[f"Feed_per_Bird_Lag{l}"] = data["Feed_per_Bird_g"].shift(l)
    data[f"Feed_Lag{l}"] = data["Feed_Consumed_kg"].shift(l)

data["LayRate_RollMean7"] = data["Lay_Rate_Pct"].shift(1).rolling(7).mean()
data["LayRate_RollStd7"] = data["Lay_Rate_Pct"].shift(1).rolling(7).std().fillna(0)
data["Feed_per_Bird_RollMean7"] = data["Feed_per_Bird_g"].shift(1).rolling(7).mean()

data["DayOfWeek"] = data["Date"].dt.dayofweek
for d in range(7):
    data[f"DOW_{d}"] = (data["DayOfWeek"] == d).astype(float)

clean_data = data.dropna().reset_index(drop=True)
print("Clean feature matrix shape:", clean_data.shape)


## 3. Model Development & Evaluation (Task 3)
We split the data chronologically (80% train = 162 days, 20% test = 41 days).
We evaluate Naive Persistence, 7-Day Moving Average, Multivariate Ridge Regression, and a Custom GBDT model.


In [ ]:
class CustomDecisionTree:
    def __init__(self, max_depth=2, min_samples_split=6):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree = None

    def fit(self, X, y, depth=0):
        if depth >= self.max_depth or len(y) < self.min_samples_split or np.var(y) < 1e-6:
            return {"val": np.mean(y)}
        n_samples, n_features = X.shape
        best_var_red = -1
        best_feat, best_thresh = None, None
        current_var = np.var(y) * n_samples
        for feat in range(n_features):
            vals = np.unique(X[:, feat])
            if len(vals) < 2:
                continue
            thresholds = np.percentile(vals, np.linspace(5, 95, 12))
            for t in thresholds:
                left_m = X[:, feat] <= t
                right_m = ~left_m
                if np.sum(left_m) == 0 or np.sum(right_m) == 0:
                    continue
                var_red = current_var - (np.var(y[left_m]) * np.sum(left_m) + np.var(y[right_m]) * np.sum(right_m))
                if var_red > best_var_red:
                    best_var_red = var_red
                    best_feat, best_thresh = feat, t
        if best_feat is None:
            return {"val": np.mean(y)}
        left_m = X[:, best_feat] <= best_thresh
        return {
            "feat": best_feat, "thresh": best_thresh,
            "left": self.fit(X[left_m], y[left_m], depth + 1),
            "right": self.fit(X[~left_m], y[~left_m], depth + 1)
        }

    def _predict_row(self, node, x):
        if "val" in node:
            return node["val"]
        return self._predict_row(node["left"], x) if x[node["feat"]] <= node["thresh"] else self._predict_row(node["right"], x)

    def predict(self, X):
        return np.array([self._predict_row(self.tree, x) for x in X])

class CustomGBDT:
    def __init__(self, n_estimators=40, learning_rate=0.08, max_depth=2):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.trees = []
        self.init_val = 0.0

    def fit(self, X, y):
        self.init_val = np.mean(y)
        y_pred = np.full(len(y), self.init_val)
        self.trees = []
        for _ in range(self.n_estimators):
            residuals = y - y_pred
            dt = CustomDecisionTree(max_depth=self.max_depth)
            dt.tree = dt.fit(X, residuals)
            y_pred += self.learning_rate * dt.predict(X)
            self.trees.append(dt)

    def predict(self, X):
        y_pred = np.full(len(X), self.init_val)
        for dt in self.trees:
            y_pred += self.learning_rate * dt.predict(X)
        return y_pred

# Chronological 80/20 train/test split
split_idx = int(len(clean_data) * 0.8)
train_df = clean_data.iloc[:split_idx].copy()
test_df = clean_data.iloc[split_idx:].copy()

features = [
    "Days_in_Lay", "Feed_per_Bird_g",
    "Feed_per_Bird_Lag1", "Feed_per_Bird_Lag2", "Feed_per_Bird_Lag3", "Feed_per_Bird_RollMean7",
    "LayRate_Lag1", "LayRate_Lag2", "LayRate_Lag3", "LayRate_Lag7",
    "LayRate_RollMean7", "LayRate_RollStd7"
]

X_train = train_df[features].values
y_train = train_df["Lay_Rate_Pct"].values
X_test = test_df[features].values
y_test = test_df["Lay_Rate_Pct"].values

# Standardize features
mean_X = X_train.mean(axis=0)
std_X = X_train.std(axis=0) + 1e-8
X_train_s = (X_train - mean_X) / std_X
X_test_s = (X_test - mean_X) / std_X

# Train Ridge
X_train_b = np.column_stack([np.ones(len(X_train_s)), X_train_s])
X_test_b = np.column_stack([np.ones(len(X_test_s)), X_test_s])
I = np.eye(X_train_b.shape[1]); I[0, 0] = 0
w_ridge = np.linalg.solve(X_train_b.T @ X_train_b + 5.0 * I, X_train_b.T @ y_train)
pred_ridge_eggs = (X_test_b @ w_ridge / 100.0) * test_df["Flock_Size"].values

# Train Custom GBDT
gbdt = CustomGBDT(n_estimators=40, learning_rate=0.08, max_depth=2)
gbdt.fit(X_train, y_train)
pred_gbdt_eggs = (gbdt.predict(X_test) / 100.0) * test_df["Flock_Size"].values

# Baselines
actual_eggs = test_df["Eggs_Collected"].values
pred_pers = test_df["Eggs_Lag1"].values
pred_roll = (test_df["LayRate_RollMean7"].values / 100.0) * test_df["Flock_Size"].values

def calc_metrics(actual, pred):
    mae = np.mean(np.abs(actual - pred))
    rmse = np.sqrt(np.mean((actual - pred)**2))
    mape = np.mean(np.abs((actual - pred) / actual)) * 100.0
    r2 = 1.0 - (np.sum((actual - pred)**2) / np.sum((actual - np.mean(actual))**2))
    return mae, rmse, mape, r2

metrics = [
    ("Naive Persistence (Lag 1)", *calc_metrics(actual_eggs, pred_pers)),
    ("7-Day Moving Average", *calc_metrics(actual_eggs, pred_roll)),
    ("Multivariate Ridge Regression", *calc_metrics(actual_eggs, pred_ridge_eggs)),
    ("Gradient Boosted Trees (GBDT)", *calc_metrics(actual_eggs, pred_gbdt_eggs))
]
res_df = pd.DataFrame(metrics, columns=["Model", "MAE (Eggs)", "RMSE (Eggs)", "MAPE (%)", "R2 Score"])
display(res_df)


In [ ]:
# Visualizations
dates = test_df["Date"].values

# 1. Actual vs. Predicted Plot
plt.figure(figsize=(10, 4.5))
plt.plot(dates, actual_eggs, label="Actual Eggs", color="#1f77b4", marker="o")
plt.plot(dates, pred_gbdt_eggs, label="GBDT Prediction", color="#2ca02c", linestyle="--")
plt.title("Actual vs. Predicted Egg Yield (Test Set)", fontweight="bold")
plt.xlabel("Date")
plt.ylabel("Eggs Collected")
plt.legend()
plt.tight_layout()
plt.show()

# 2. Residual Distribution
residuals = actual_eggs - pred_gbdt_eggs
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].axhline(0, color="red", linestyle="--")
axes[0].plot(dates, residuals, marker="s", color="#9467bd")
axes[0].set_title("Residual Error Over Time")
axes[0].set_ylabel("Error (Eggs)")
axes[1].hist(residuals, bins=10, color="#17becf", edgecolor="black")
axes[1].set_title("Residual Error Histogram")
axes[1].set_xlabel("Error (Eggs)")
plt.tight_layout()
plt.show()


In [ ]:
# 14-Day Forward Operational Forecast (Sep 28 to Oct 11, 2026)
last_date = clean_data["Date"].max()
last_flock = int(clean_data["Flock_Size"].iloc[-1])
daily_feed_kg = 194.0
feed_per_bird = (daily_feed_kg * 1000.0) / last_flock

sim_rates = list(clean_data["Lay_Rate_Pct"].values)
sim_feeds = list(clean_data["Feed_per_Bird_g"].values)

forecast_rows = []
for i in range(1, 15):
    f_date = last_date + pd.Timedelta(days=i)
    days_in_lay = len(clean_data) + i - 1
    row_vals = [
        days_in_lay, feed_per_bird, sim_feeds[-1], sim_feeds[-2], sim_feeds[-3],
        np.mean(sim_feeds[-7:]), sim_rates[-1], sim_rates[-2], sim_rates[-3],
        sim_rates[-7:], np.mean(sim_rates[-7:]), np.std(sim_rates[-7:])
    ]
    x_vec = np.array([row_vals])
    pred_rate = gbdt.predict(x_vec)[0]
    pred_eggs = int(round((pred_rate / 100.0) * last_flock))
    trays = round(pred_eggs / 30.0, 1)
    rev = int(round((pred_eggs / 30.0) * 10000.0))
    cost = int(round(daily_feed_kg * 1450.0))
    margin = rev - cost
    forecast_rows.append({
        "Date": f_date.strftime("%Y-%m-%d"),
        "Flock_Size": last_flock,
        "Feed_kg": daily_feed_kg,
        "Pred_Lay_Rate_%": round(pred_rate, 2),
        "Pred_Eggs": pred_eggs,
        "Pred_Trays": trays,
        "Est_Revenue_UGX": rev,
        "Est_Feed_Cost_UGX": cost,
        "Est_Gross_Margin_UGX": margin
    })
    sim_rates.append(pred_rate)
    sim_feeds.append(feed_per_bird)

fc_df = pd.DataFrame(forecast_rows)
display(fc_df)
